<a href="https://colab.research.google.com/github/mttomar/big-data-analytics/blob/main/pyspark_friend_recommendations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark

In [2]:
from pyspark import SparkContext
from pyspark.sql import SQLContext

In [3]:
sc = SparkContext.getOrCreate()
sqlContext = SQLContext(sc)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [4]:
from google.colab import files
uploaded = files.upload()

Saving Friends.txt to Friends.txt


In [5]:
input_file = sc.textFile('Friends.txt')

input_file.take(10)

['0\t1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94',
 '1\t0,5,20,135,2409,8715,8932,10623,12347,12846,13840,13845,14005,20075,21556,22939,23520,28193,29724,29791,29826,30691,31232,31435,32317,32489,34394,35589,35605,35606,35613,35633,35648,35678,38737,43447,44846,44887,49226,49985,623,629,4999,6156,13912,14248,15190,17636,19217,20074,27536,29481,29726,29767,30257,33060,34250,34280,34392,34406,34418,34420,34439,34450,34651,45054,49592',
 '2\t0,117,135,1220,2755,12453,24539,24714,41456,45046,49927,6893,13795,16659,32828,41878',
 '3\t0,12,41,55,1532,12636,13185,27552,38737',
 '4\t0,8,14,15,18,27,72,80,15326,19068,19079,24596,42697,46126,74,77,33269,38792,38822',
 '5\t0,1,20,2022,22939,23527,30257,32503,35633,41457,43262,44846,49574,31140,32828',
 '6\t0,21,98,2203,32

In [6]:
adj_list = input_file \
    .map(lambda line: line.strip().split("\t")) \
    .map(lambda x: (int(x[0]), [] if len(x) == 1 or x[1] == "" else [int(i) for i in x[1].split(",")]))

adj_list.take(5)

[(0,
  [1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94]),
 (1,
  [0,
   5,
   20,
   135,
   2409,
   8715,
   8932,
   10623,
   12347,
   12846,
   13840,
   13845,
   14005,
   20075,
   21556,
   22939,
   23520,
   28193,
   29724,
   29791,
   29826,
   30691,
   31232,
   31435,
   32317,
   32489,
   34394,
   35589,
   35605,
   35606,
   35613,
   35633,
   35648,
   35678,
   38737,
   43

In [7]:
from itertools import combinations

listA = adj_list.flatMap(
    lambda x: [((a, b), 1) for a, b in combinations(sorted(x[1]), 2)] +
              [((b, a), 1) for a, b in combinations(sorted(x[1]), 2)]
)

In [8]:
listA.take(10)

[((1, 2), 1),
 ((1, 3), 1),
 ((1, 4), 1),
 ((1, 5), 1),
 ((1, 6), 1),
 ((1, 7), 1),
 ((1, 8), 1),
 ((1, 9), 1),
 ((1, 10), 1),
 ((1, 11), 1)]

In [9]:
listA.count()

22909086

In [10]:
listA_counts = listA.reduceByKey(lambda a, b: a + b)

In [11]:
listA_counts.take(10)

[((2644, 1234), 1),
 ((1120, 17636), 1),
 ((2752, 10208), 1),
 ((1394, 34250), 1),
 ((29399, 40185), 2),
 ((2795, 34329), 1),
 ((19432, 34568), 4),
 ((23615, 21243), 4),
 ((32489, 11107), 1),
 ((14209, 34319), 16)]

In [12]:
listB = adj_list.flatMap(
    lambda x: [((x[0], f), 1) for f in x[1]]
)

In [13]:
listB.take(10)

[((0, 1), 1),
 ((0, 2), 1),
 ((0, 3), 1),
 ((0, 4), 1),
 ((0, 5), 1),
 ((0, 6), 1),
 ((0, 7), 1),
 ((0, 8), 1),
 ((0, 9), 1),
 ((0, 10), 1)]

In [14]:
filtered = listA_counts.subtractByKey(listB)

In [15]:
filtered = filtered.cache()

In [16]:
filtered.take(10)

[((18148, 8706), 1),
 ((32210, 19920), 1),
 ((13996, 8686), 1),
 ((23633, 14205), 7),
 ((1137, 3789), 1),
 ((40739, 43543), 1),
 ((14112, 12050), 1),
 ((2141, 27577), 1),
 ((35630, 26440), 2),
 ((35659, 29723), 1)]

In [17]:
by_user = filtered.map(lambda x: (x[0][0], (x[0][1], x[1])))

In [18]:
by_user.take(10)

[(18148, (8706, 1)),
 (32210, (19920, 1)),
 (13996, (8686, 1)),
 (23633, (14205, 7)),
 (1137, (3789, 1)),
 (40739, (43543, 1)),
 (14112, (12050, 1)),
 (2141, (27577, 1)),
 (35630, (26440, 2)),
 (35659, (29723, 1))]

In [19]:
result = by_user.groupByKey().map(
    lambda x: (
        x[0],
        sorted(list(x[1]), key=lambda y: (-y[1], y[0]))[:10]
    )
)

In [20]:
result.take(10)

[(18148,
  [(18147, 4),
   (1670, 3),
   (1705, 3),
   (2698, 3),
   (2711, 3),
   (49543, 3),
   (95, 2),
   (204, 2),
   (385, 2),
   (1220, 2)]),
 (13996,
  [(23690, 3),
   (25224, 3),
   (9998, 2),
   (12625, 2),
   (13818, 2),
   (16899, 2),
   (19443, 2),
   (21556, 2),
   (23305, 2),
   (25186, 2)]),
 (14112,
  [(35630, 5),
   (19, 4),
   (7476, 4),
   (14064, 4),
   (15175, 4),
   (13870, 3),
   (14014, 3),
   (14056, 3),
   (14065, 3),
   (14152, 3)]),
 (25224,
  [(27679, 13),
   (18916, 11),
   (27549, 10),
   (34299, 10),
   (10099, 9),
   (19, 8),
   (15186, 8),
   (21873, 8),
   (25200, 8),
   (27536, 8)]),
 (27388,
  [(27888, 9),
   (2233, 5),
   (18591, 5),
   (34875, 5),
   (42692, 5),
   (2118, 4),
   (2196, 4),
   (2253, 4),
   (2259, 4),
   (2264, 4)]),
 (27636,
  [(25186, 4),
   (12, 2),
   (1135, 2),
   (18916, 2),
   (19365, 2),
   (25215, 2),
   (25224, 2),
   (27555, 2),
   (27564, 2),
   (27585, 2)]),
 (120,
  [(182, 4),
   (185, 3),
   (192, 3),
   (17, 2),
  

In [21]:
formatted = result.map(
    lambda x: str(x[0]) + "\t" + ",".join([str(i[0]) for i in x[1]])
)

In [22]:
formatted.take(10)

['18148\t18147,1670,1705,2698,2711,49543,95,204,385,1220',
 '13996\t23690,25224,9998,12625,13818,16899,19443,21556,23305,25186',
 '14112\t35630,19,7476,14064,15175,13870,14014,14056,14065,14152',
 '25224\t27679,18916,27549,34299,10099,19,15186,21873,25200,27536',
 '27388\t27888,2233,18591,34875,42692,2118,2196,2253,2259,2264',
 '27636\t25186,12,1135,18916,19365,25215,25224,27555,27564,27585',
 '120\t182,185,192,17,99,103,115,122,128,149',
 '132\t7812,18991,17,96,97,98,99,100,101,102',
 '136\t34816,48809,48810,48812,48814,48815,48816,48817,48818,48820',
 '32628\t32629,32631,19,439,667,975,1085,1100,1150,1689']

In [23]:
all_users = adj_list.map(lambda x: x[0])
users_with_recs = formatted.map(lambda line: int(line.split("\t")[0]))
missing_users = all_users.subtract(users_with_recs).map(lambda u: str(u) + "\t")
final_output = formatted.union(missing_users).sortBy(lambda x: int(x.split("\t")[0]))

In [ ]:
final_output.take(20)

In [ ]:
output_lines = final_output.collect()

with open("recommendations.txt", "w") as f:
    for line in output_lines:
        f.write(line + "\n")

In [ ]:
required_users = {18, 87, 480, 512, 3971, 38283, 49772, 1152}

required_output = final_output.filter(
    lambda line: int(line.split("\t")[0]) in required_users
).collect()

for line in sorted(required_output, key=lambda x: int(x.split("\t")[0])):
    print(line)

In [ ]:
my_extra_user = 1142

extra_output = final_output.filter(
    lambda line: int(line.split("\t")[0]) == my_extra_user
).collect()

for line in extra_output:
    print(line)